# Predictive Maintenance Pipeline

## Objective
This notebook is a **client léger** of the `indusense` package: loading, anti-leakage exclusions and
temporal split are handled by `indusense.ml.data` (driven by `configs/preprocessing.yaml`), not
redefined here. This notebook adds a linear-model-oriented step (median imputation + standardization)
on top of that shared logic.

1. **Load** the gold dataset via `indusense.ml.data.load_gold_dataset()`.
2. **Temporal split** via `indusense.ml.data.temporal_split()` (train < validation < test).
3. **Preprocess**: impute NaN values (median) and standardize features for linear models.

### Dataset Overview
- **Source**: `RAW_DIR / gold_dataset.parquet` (see `indusense.common.config`, no hardcoded path)
- **Key Columns**:
  - `machine_id_std`: Machine identifier (e.g., `MACH-01`)
  - `window_start`: Timestamp for the start of the measurement window
  - `label_failure_next_24h`: Target variable for 24h horizon
  - `split_set`: Predefined temporal split (train/validation/test)


In [ ]:
# Imports
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from indusense.ml.data import load_gold_dataset, temporal_split
from indusense.common.config import MODELS_DIR, OUTPUT_DIR


In [6]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split  # For reference (not used here, temporal split is used)

---
## Step 1: Load the Data (`indusense.ml.data.load_gold_dataset`)

Chargement, tri par machine/temps et exclusions anti-fuite déjà appliqués — la liste des colonnes
exclues vit dans `configs/preprocessing.yaml` (clé `ml.exclude_columns`), pas dans ce notebook.


In [7]:
df, feats, horizon = load_gold_dataset()

print('Dataset shape:', df.shape)
print(f'Features conservées (anti-fuite exclue) : {len(feats)}')
print('First 5 rows:')
display(df.head())


Dataset shape: (134280, 100)
First 5 rows:


,machine_id_std,window_start,temp_mean_1h,temp_max_1h,pressure_mean_1h,pressure_max_1h,voltage_mean_1h,voltage_max_1h,rotation_mean_1h,rotation_max_1h,...,maintenance_count_prev_30d,future_incident_count_6h,label_failure_next_6h,future_incident_count_12h,label_failure_next_12h,future_incident_count_24h,label_failure_next_24h,future_incident_count_48h,label_failure_next_48h,split_set
0,MACH-01,2025-06-01 00:00:00,46.340,46.340,198.203,198.203,227.568,227.568,1541.787,1541.787,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
1,MACH-01,2025-06-01 01:00:00,48.762,48.762,198.295,198.295,227.480,227.480,1537.860,1537.860,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
2,MACH-01,2025-06-01 02:00:00,51.352,51.352,199.545,199.545,228.680,228.680,1584.660,1584.660,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
3,MACH-01,2025-06-01 03:00:00,49.512,49.512,201.641,201.641,228.440,228.440,1588.960,1588.960,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train
4,MACH-01,2025-06-01 04:00:00,51.982,51.982,200.157,200.157,227.840,227.840,1548.660,1548.660,...,0,0.0,False,0.0,False,0.0,False,0.0,False,train


First 10 rows (machine and time):


,machine_id_std,window_start
0,MACH-01,2025-06-01 00:00:00
1,MACH-01,2025-06-01 01:00:00
2,MACH-01,2025-06-01 02:00:00
3,MACH-01,2025-06-01 03:00:00
4,MACH-01,2025-06-01 04:00:00
5,MACH-01,2025-06-01 05:00:00
6,MACH-01,2025-06-01 06:00:00
7,MACH-01,2025-06-01 07:00:00
8,MACH-01,2025-06-01 08:00:00
9,MACH-01,2025-06-01 09:00:00


---
## Step 2: Anti-Leakage Exclusions (déjà appliquées)

La liste des colonnes exclues (identifiants, colonnes de fuite temporelle, indicateurs
d'incidents, colonnes de maintenance, `split_set`) est centralisée dans
`configs/preprocessing.yaml` — `feats` ci-dessus en tient déjà compte.


In [9]:
print(f'Nombre de features : {len(feats)}')
print('Features :', feats)


Number of remaining columns: 62
Remaining columns:
['temp_mean_1h', 'temp_max_1h', 'pressure_mean_1h', 'pressure_max_1h', 'voltage_mean_1h', 'voltage_max_1h', 'rotation_mean_1h', 'rotation_max_1h', 'pieces_produced_sum_1h', 'temp_mean_6h', 'temp_max_6h', 'temp_std_6h', 'pressure_mean_6h', 'pressure_max_6h', 'pressure_std_6h', 'voltage_mean_6h', 'voltage_max_6h', 'voltage_std_6h', 'rotation_mean_6h', 'rotation_max_6h', 'rotation_std_6h', 'temp_mean_12h', 'temp_max_12h', 'temp_std_12h', 'pressure_mean_12h', 'pressure_max_12h', 'pressure_std_12h', 'voltage_mean_12h', 'voltage_max_12h', 'voltage_std_12h', 'rotation_mean_12h', 'rotation_max_12h', 'rotation_std_12h', 'temp_mean_24h', 'temp_max_24h', 'temp_std_24h', 'pressure_mean_24h', 'pressure_max_24h', 'pressure_std_24h', 'voltage_mean_24h', 'voltage_max_24h', 'voltage_std_24h', 'rotation_mean_24h', 'rotation_max_24h', 'rotation_std_24h', 'temp_trend_6h', 'pressure_trend_6h', 'voltage_trend_6h', 'rotation_trend_6h', 'temp_zscore_24h', 'te

---
## Step 3: Build the Target Variable (y)

Select a **horizon** (e.g., `label_failure_next_24h`) and create a **binary target variable** (`y`).

### Notes:
- The target variable is derived from the selected horizon column.
- Values are converted from boolean to **0/1** (0: no failure, 1: failure).

In [10]:
y = df[horizon].astype(int)

print('Target distribution:')
print(y.value_counts())

failure_ratio = y.mean()
print(f'Failure ratio: {failure_ratio:.4f} ({failure_ratio * 100:.2f}%)')


Target distribution:
label_failure_next_24h
0    111732
1     22548
Name: count, dtype: int64
Failure ratio: 0.1679 (16.79%)


---
## Step 3: Temporal Split (`indusense.ml.data.temporal_split`)

Split positionnel via la colonne `split_set` déjà fournie dans le Gold — jamais un split
aléatoire, pour respecter l'ordre temporel (pas de fuite du futur vers le passé).


In [11]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = temporal_split(df, feats, horizon)

for name, (X, y_) in [('Train', (X_train, y_train)), ('Validation', (X_val, y_val)), ('Test', (X_test, y_test))]:
    print(f'{name} set:')
    print(f'  Features: {X.shape}, Target: {y_.shape}')
    print(f'  Failure ratio: {y_.mean():.4f}')


Train set:
  Features: (93990, 62), Target: (93990,)
  Failure ratio: 0.1660
Validation set:
  Features: (20145, 62), Target: (20145,)
  Failure ratio: 0.1724
Test set:
  Features: (20145, 62), Target: (20145,)
  Failure ratio: 0.1726


---
## Step 4: Impute NaN Values (Median) and Standardize

Étape spécifique à ce notebook (pas dans le package) : les modèles à arbres (XGBoost) n'ont pas
besoin d'imputation/standardisation, mais un modèle linéaire (régression logistique, SVM) en a besoin.


In [12]:
# Create a pipeline for imputation and standardization
preprocessing_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Impute NaN with median
    ('scaler', StandardScaler())  # Standardize features (mean=0, std=1)
])

# Fit and transform the training data
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Transform validation and test data (using the same pipeline)
X_val_processed = preprocessing_pipeline.transform(X_val)
X_test_processed = preprocessing_pipeline.transform(X_test)

# Verify no NaN values remain
print('NaN values after preprocessing:')
print(f'  X_train: {pd.isna(X_train_processed).sum().sum()}')
print(f'  X_val: {pd.isna(X_val_processed).sum().sum()}')
print(f'  X_test: {pd.isna(X_test_processed).sum().sum()}')

# Verify standardization (mean and std for first 5 features)
print(f'Standardization check (first 5 features):')
for i in range(5):
    print(f'  Feature {i}: mean={X_train_processed[:, i].mean():.4f}, std={X_train_processed[:, i].std():.4f}')

NaN values after preprocessing:
  X_train: 0
  X_val: 0
  X_test: 0
Standardization check (first 5 features):
  Feature 0: mean=0.0000, std=1.0000
  Feature 1: mean=0.0000, std=1.0000
  Feature 2: mean=0.0000, std=1.0000
  Feature 3: mean=0.0000, std=1.0000
  Feature 4: mean=-0.0000, std=1.0000


---
## Summary

### Pipeline Recap:
1. ✅ **Loaded** via `indusense.ml.data.load_gold_dataset()` (tri + exclusions anti-fuite).
2. ✅ **Temporal split** via `indusense.ml.data.temporal_split()`.
3. ✅ **Built the target variable** (`y`) from the configured horizon.
4. ✅ **Preprocessed features**: imputed NaN values (median) and standardized.

### Outputs (dans `MODELS_DIR`/`OUTPUT_DIR`, jamais un chemin en dur):
- `X_train_processed`, `y_train`: Training set (preprocessed)
- `X_val_processed`, `y_val`: Validation set (preprocessed)
- `X_test_processed`, `y_test`: Test set (preprocessed)
- `preprocessing_pipeline`: Reusable pipeline for new data


In [14]:
import joblib

joblib.dump(preprocessing_pipeline, MODELS_DIR / 'preprocessing_pipeline.joblib')

pd.DataFrame(X_train_processed).to_parquet(OUTPUT_DIR / 'X_train_processed.parquet')
pd.DataFrame(X_val_processed).to_parquet(OUTPUT_DIR / 'X_val_processed.parquet')
pd.DataFrame(X_test_processed).to_parquet(OUTPUT_DIR / 'X_test_processed.parquet')

pd.DataFrame(y_train).to_parquet(OUTPUT_DIR / 'y_train.parquet')
pd.DataFrame(y_val).to_parquet(OUTPUT_DIR / 'y_val.parquet')
pd.DataFrame(y_test).to_parquet(OUTPUT_DIR / 'y_test.parquet')

print(f'Processed data saved to {MODELS_DIR} / {OUTPUT_DIR}.')


Processed data saved to disk.
